In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quantum Portfolio Optimizer: funkce Qiskit od Global Data Quantum


> **Note:** Qiskit Functions jsou experimentální funkce dostupné pouze uživatelům plánů IBM Quantum&reg; Premium Plan, Flex Plan a On-Prem (prostřednictvím IBM Quantum Platform API) Plan. Jsou ve stavu náhledového vydání a mohou se změnit.
# Přehled
Quantum Portfolio Optimizer je funkce Qiskit, která řeší problém dynamické optimalizace portfolia – standardní problém ve financích, jehož cílem je pravidelně přebalancovávat investice napříč sadou aktiv tak, aby se maximalizovaly výnosy a minimalizovala rizika. Díky nasazení špičkových technik kvantové optimalizace tato funkce celý proces zjednodušuje, takže z jejích výhod při hledání optimálních investičních trajektorií mohou těžit i uživatelé bez znalostí kvantového počítání. Tento nástroj je ideální pro portfolio manažery, výzkumníky v oblasti kvantitativních financí a individuální investory a umožňuje zpětné testování obchodních strategií při optimalizaci portfolia.
# Popis funkce
Funkce Quantum Portfolio Optimizer využívá algoritmus Variational Quantum Eigensolver (VQE) k řešení problému Quadratic Unconstrained Binary Optimization (QUBO), čímž řeší problémy dynamické optimalizace portfolia. Stačí poskytnout data o cenách aktiv a definovat investiční omezení; funkce pak spustí proces kvantové optimalizace, který vrátí sadu optimalizovaných investičních trajektorií.

Proces sestává ze čtyř hlavních fází. Nejprve jsou vstupní data mapována na problém kompatibilní s kvantovým počítáním: je zkonstruováno QUBO problému dynamické optimalizace portfolia a transformováno do kvantového operátoru (Isingův hamiltonián). Dále jsou vstupní problém a algoritmus VQE přizpůsobeny pro spuštění na kvantovém hardwaru. Algoritmus VQE je pak spuštěn na kvantovém hardwaru a nakonec jsou výsledky post-zpracovány tak, aby poskytly optimální investiční trajektorie. Systém také zahrnuje post-zpracování zohledňující šum (založené na [SQD](/guides/qiskit-addons-sqd)) pro maximalizaci kvality výstupu.

Tato funkce Qiskit vychází z [publikovaného článku](https://arxiv.org/abs/2412.19150) od Global Data Quantum.
![Vizualizace pracovního postupu funkce](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Vstup
Vstupní argumenty funkce jsou popsány v následující tabulce. Musí být poskytnuta data aktiv a ostatní specifikace problému; navíc lze zahrnout nastavení VQE pro přizpůsobení procesu optimalizace.
| **Název** | **Typ** | **Popis** | **Povinný** | **Výchozí hodnota** | **Příklad** |
|---|---|---|---|---|---|
| assets | json | Slovník s cenami aktiv | Ano | - | - |
| qubo_settings | json | Nastavení QUBO | Ano | - | Viz příklady v tabulce níže |
| ansatz_settings | json | Nastavení ansatzu | Ne | `None` | Viz příklady v tabulce níže. |
| optimizer_settings | json | Nastavení optimalizátoru | Ne | `None` | Viz příklady v tabulce níže. |
| backend | str | Název backendového QPU | Ne | - | `"ibm_torino"` |
| previous_session_id | list of str | Seznam ID Session pro načtení dat z předchozích spuštění[(*)](#1-note) | Ne | Prázdný seznam | `["session_id_1", "session_id_2"]` |
| apply_postprocess | bool | Použít post-zpracování SQD zohledňující šum | Ne | `True` | `True` |
| tags | list of strings | Seznam štítků pro identifikaci experimentu | Ne | Prázdný seznam | `["optimization", "quantum_computing"]` |
<span id="1-note"></span>
*Chceš-li pokračovat v provádění nebo načíst úlohy zpracované v jedné či více předchozích Session, musíš do parametru `previous_session_id` předat seznam ID Session. To je zvláště užitečné v případech, kdy optimalizační úloha nedokončila kvůli chybě v průběhu procesu a je třeba dokončit provádění. K tomu musíš poskytnout stejné argumenty jako při prvotním spuštění spolu se seznamem `previous_session_id`, jak je popsáno.*
> **Caution:** Načítání dat pro předchozí Session (za účelem obnovení optimalizace) může trvat až jednu hodinu.
## `assets`
Data musí být strukturována jako objekt JSON, který uchovává informace o závěrečných cenách finančních aktiv k určitým datům. Formát je následující:

- Primární klíč (řetězec): Název nebo tickerový symbol finančního aktiva (například „8801.T").
- Sekundární klíč (řetězec): Datum ve formátu YYYY-MM-DD.
- Hodnota (číslo): Závěrečná cena aktiva k danému datu. Ceny lze zadat normalizované i nenormalizované.

*Všimni si, že všechny slovníky musí mít stejný sekundární klíč (data). Pokud určité aktivum nemá datum, které mají ostatní, musí být data doplněna tak, aby byla zajištěna konzistence. Toho lze docílit například použitím poslední sledované závěrečné ceny daného aktiva.*
### Příklad
``` py
{
    "8801.T": {
        "2023-01-01": 2374.0,
        "2023-01-02": 2374.0,
        "2023-01-03": 2374.0,
        "2023-01-04": 2356.5,
        ...
    },
    "AAPL": {
        "2023-01-01": 145.2,
        "2023-01-02": 146.5,
        "2023-01-03": 147.3,
        "2023-01-04": 148.1,
        ...
    },
    ...
}
```

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

> **Note:** Data aktiv musí obsahovat alespoň závěrečné ceny v ``(nt+1) * dt`` (viz sekci vstupu [`qubo_settings`](#qubo_settings)) časových razítkách (například dnech).
## `qubo_settings`
Následující tabulka popisuje klíče slovníku `qubo_settings`. Slovník vytvoř tak, že zadáš počet časových kroků `nt`, počet rozlišovacích Qubitů `nq` a `max_investment` – nebo změň jiné výchozí hodnoty.
| Název               | Typ     | Popis                                                                       | Povinný | Výchozí | Příklad |
|---------------------|---------|-----------------------------------------------------------------------------|---------|---------|---------|
| nt                  | int     | Počet časových kroků                                                        | Ano     | -       | 4       |
| nq                  | int     | Počet rozlišovacích Qubitů                                                  | Ano     | -       | 4       |
| max_investment      | float   | Maximální počet investovaných měnových jednotek napříč všemi aktivy         | Ano     | -       | 10      |
| dt*                  | int     | Časové okno uvažované v každém časovém kroku. Jednotka odpovídá časovým intervalům mezi klíči v datech aktiv | Ne      | 30      | -       |
| risk_aversion       | float   | Koeficient averze k riziku                                                  | Ne      | 1000    | -       |
| transaction_fee     | float   | Koeficient transakčního poplatku                                            | Ne      | 0.01    | -       |
| restriction_coeff   | float   | Lagrangeův multiplikátor používaný k vynucení omezení problému v QUBO formulaci | Ne  | 1       | -       |
## `ansatz_settings`
Chceš-li změnit výchozí možnosti, vytvoř slovník pro parametr `ansatz_settings` s následujícími klíči. Výchozí ansatz je nastaven na `"real_amplitudes"` a obě doplňkové možnosti (viz následující tabulka) jsou nastaveny na `False`.
| Název                 | Typ      | Popis                                                                       | Povinný | Výchozí |
|-----------------------|----------|-----------------------------------------------------------------------------|---------|---------|
| ansatz[*](#single-star)                | str      | Ansatz, který se má použít                                  | Ne      | `"real_amplitudes"` |
| multiple_passmanager[**](#double-star)  | bool     | Povoluje podprogram multiple passmanager (není k dispozici pro Tailored ansatz) | Ne | `False` |
| dd_enable   | bool     | Přidává dynamické oddělování                                                | Ne      | `False` |

<span id="single-star"></span>
\* Dostupné ansatze
- `real_amplitudes`
- `cyclic`
- `optimized_real_amplitudes`
- `tailored` (Pouze pro Backend `ibm_torino`, 7 aktiv, 4 časové kroky a 4 rozlišovací Qubity)

<span id="double-star"></span>
** Pokud je ``multiple_passmanager`` nastaven na ``False``, funkce používá výchozí správce průchodů Qiskit s `optimization_level=3`. Pokud je nastaven na ``True``, podprogram ``multiple_passmanager`` porovná tři správce průchodů: předchozí výchozí správce průchodů Qiskit, správce průchodů mapující Qubity na řetězec nejbližších sousedů QPU a služby [AI transpileru](/guides/ai-transpiler-passes). Poté je vybrán správce průchodů s odhadovanou nižší kumulativní chybou.
## `optimizer_settings`
Tento parametr je slovník s některými nastavitelnými možnostmi optimalizačního procesu.
| Název              | Typ    | Popis                                            | Povinný | Výchozí               |
|--------------------|--------|--------------------------------------------------|---------|------------------------|
| primitive_options  | json   | Nastavení primitivu                              | Ne      | -                      |
| optimizer          | str    | Vybraný klasický optimalizátor                   | Ne      | `"differential_evolution"` |
| optimizer_options  | json   | Konfigurace optimalizátoru                       | Ne      | -                      |
> **Note:** V současnosti je jedinou dostupnou možností optimalizátoru `"differential_evolution"`.

Pod klíči `primitive_options` a `optimizer_options` nastavujeme slovníky s následujícími parametry:
### `primitive_options`
| Název              | Typ    | Popis                                      | Povinný | Výchozí                    | Příklad                    |
|--------------------|--------|--------------------------------------------|---------|----------------------------|----------------------------|
| sampler_shots     | int    | Počet výstřelů Sampleru.                   | Ne      | 100000                     | -                          |
| estimator_shots   | int    | Počet výstřelů Estimatoru.                 | Ne      | 25000                      | -                          |
| estimator_precision | float | Požadovaná přesnost očekávané hodnoty. Je-li zadána, bude místo `estimator_shots` použita přesnost. | Ne | `None` | 0.015625 · (1 / sqrt(4096)) |
| max_time          | int nebo str | Maximální doba, po kterou může runtime Session zůstat otevřená, než bude násilně uzavřena. Lze zadat v sekundách (int) nebo jako řetězec, například `"2h 30m 40s"`. Musí být menší než systémem stanovené maximum. | Ne | `None` | `"1h 15m"` |
### `optimizer_options`
| Název              | Typ      | Popis                                     | Povinný | Výchozí       |
|--------------------|----------|-------------------------------------------|---------|---------------|
| num_generations   | int      | Počet generací                            | Ne      | 20            |
| population_size   | int      | Velikost populace                         | Ne      | 20            |
| mutation_range    | list     | Maximální a minimální mutační faktor      | Ne      | [0, 0.25]     |
| recombination     | float    | Rekombinační faktor                       | Ne      | 0.4           |
| max_parallel_jobs | int      | Maximální počet úloh QPU prováděných paralelně | Ne | 3             |
| max_batchsize | int      | Maximální velikost dávky                  | Ne      | 200           |
> **Note:** - Počet generací vyhodnocených diferenciální evolucí je `num_generations` + 1, protože je zahrnuta počáteční populace.
> - Celkový počet Circuitů se vypočítá jako `(num_generations + 1) * population_size`.
> - Větší velikost populace a více generací obecně zlepšuje kvalitu výsledků optimalizace. Nedoporučuje se však překročit velikost populace 120 a počet generací větší než 20 (například ``120 * 21 = 2520`` Circuitů celkem), protože by to vygenerovalo nadměrné množství Circuitů, jejichž zpracování může být výpočetně nákladné a časově náročné.
> 
> - Funkce ti umožňuje obnovit předchozí optimalizaci a vždy je možné zvýšit počet generací (zadáním stejného vstupu kromě `previous_session_id` a zvýšeného ``num_generations``).
> **Note:** Zajisti soulad s limity úloh Qiskit Runtime.
> 
> - Sampler: `sampler_shots <= 10_000_000`.
> - Estimator: `max_batchsize * estimator_shots * observable_size <= 10_000_000` (pro tuto funkci všechny členy pozorovatelné veličiny komutují, takže `observable_size=1`).
> 
> Více informací najdeš v průvodci [Limity úloh](/guides/job-limits).
# Výstup
Funkce vrací dva slovníky: slovník `"result"`, který obsahuje nejlepší výsledky optimalizace, včetně optimálního řešení a souvisejících minimálních nákladů cílové funkce; a `"metadata"`, s daty ze všech výsledků získaných během procesu optimalizace, spolu s jejich příslušnými metrikami.

První slovník se zaměřuje na nejlépe fungující řešení, zatímco druhý poskytuje podrobné informace o všech řešeních, včetně nákladů cílové funkce a dalších relevantních metrik.

## Výstupní slovníky:
| **Název**   | **Typ**                      | **Popis**                                                                       | **Příklad**  |
|-------------|------------------------------|---------------------------------------------------------------------------------|--------------|
| result      | dict[str, dict[str, float]]  | Obsahuje investiční strategii v čase; každý časový okamžik je mapován na váhy investic pro jednotlivá aktiva (každá váha je výše investice normalizovaná celkovou investovanou částkou). | `{'time_1': {'asset_1': 0.2, 'asset_2': 0.3, ...}, ...}` |
| metadata    | dict[str, Any]               | Data vygenerovaná během analýzy, včetně řešení, nákladů a metrik.               | Viz příklady níže |
### Popis slovníku `metadata`
| **Název**                | **Typ**               | **Popis**                                                                                           | **Příklad**                  |
|--------------------------|-----------------------|-----------------------------------------------------------------------------------------------------|------------------------------|
| session_id               | str                   | Jedinečný identifikátor relace IBM Quantum.                                     | `"d0h30qjvpqf00084fgw0"`        |
| all_samples_metrics | dict                 | Slovník obsahující různé metriky pro každý postprocesovaný vzorek, jako jsou náklady nebo omezení. | Viz popis níže        |
| sampler_counts           | dict[str, int]        | Slovník, kde klíče jsou bitové reprezentace vzorkovaných řešení a hodnoty jsou jejich počty. | `{"101010": 3, "111000": 1}` |
| asset_order              | list[str]             | Seznam odpovídajícího pořadí investic do aktiv v každém časovém kroku v rámci investičních strategií. | `["Asset_0", "Asset_1", "Asset_3"]`     |
| QUBO                     | list[list[float]]     | QUBO matice problému.                                                                               | `[[-6.96e-01, 5.81e-01, -1.26e-02, 0.00e+00], ...]`     |
| resource_summary           | dict[str, dict[str, float]] | Přehled časů využití CPU a QPU (v sekundách) v různých fázích procesu.                | `{'RUNNING: EXECUTING_QPU': {'CPU_TIME': 412.84, 'QPU_TIME': 87.22}, ...}` |
### Popis slovníku `all_samples_metrics`
| **Název**               | **Typ**        | **Popis**                                                                                                        | **Příklad**                  |
|-------------------------|----------------|------------------------------------------------------------------------------------------------------------------|------------------------------|
| investment_trajectories | list[list]     | Investiční strategie odvozené z dekódovaných kvantových stavů. | `[[1, 2, 2], [1, 2, 1]]`     |

| counts                  | list[int]      | Počet, kolikrát byla každá investiční trajektorie vzorkována. Index odpovídá `investment_trajectories`.         | `[5, 3]`                     |
| objective_costs         | list[float]    | Hodnota účelové funkce pro každou investiční trajektorii, seřazená od nejnižší po nejvyšší.                    | `[0.98, 1.25]`               |
| sharpe_ratios           | list[float]    | Rizikově upravený výkon (Sharpeho poměr) pro každou investiční trajektorii. Indexy jsou zarovnány.             | `[1.1, 0.7]`                 |
| returns                 | list[float]    | Očekávaný výnos pro každou investiční trajektorii. Indexy jsou zarovnány.                                       | `[0.15, 0.10]`               |
| rest_breaches           | list[float]    | Maximální odchylka od omezení v rámci každé investiční trajektorie. Indexy jsou zarovnány.                      | `[0.0, 0.25]`                |
| transaction_costs       | list[float]    | Odhadované transakční náklady spojené s každou investiční trajektorií. Indexy jsou zarovnány.                   | `[0.01, 0.02]`               |
# Začínáme
Autentizuj se pomocí svého [API klíče](https://quantum.cloud.ibm.com/) a vyber Qiskit Function takto. (Tento úryvek předpokládá, že sis již [uložil(a) účet](/guides/functions#install-qiskit-functions-catalog-client) do svého lokálního prostředí.)

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

## Příklad: Dynamická optimalizace portfolia se sedmi aktivy
Tento příklad ukazuje, jak spustit funkci dynamické optimalizace portfolia (DPO) a upravit její nastavení pro optimální výkon. Zahrnuje podrobné kroky pro doladění parametrů s cílem dosáhnout požadovaných výsledků.

V tomto případě se jedná o sedm aktiv, čtyři časové kroky a čtyři rozlišovací qubity, což dohromady vyžaduje 112 qubitů.
### 1. Načti aktiva zahrnutá v portfoliu.
Pokud jsou všechna aktiva v portfoliu uložena ve složce na určité cestě, můžeš je načíst do objektu `pandas.DataFrame` a převést do formátu `dict` pomocí následující funkce.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

V tomto příkladu jsme použili aktiva [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y) a [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). Následující obrázek znázorňuje data použitá v tomto příkladu – vývoj denních závěrečných cen aktiv od 1. ledna do 1. září 2023.

V tomto příkladu jsme dny bez obchodování doplnili závěrečnou cenou z předchozího dostupného data, abychom zajistili jednotnost dat napříč daty. Tento krok aplikujeme proto, že vybraná aktiva pocházejí z různých trhů s odlišnými obchodními dny, a proto je nezbytné dataset standardizovat pro konzistenci.
![Vizualizace historických dat aktiv](../docs/images/guides/global-data-quantum-optimizer/assets.avif)

### 2. Definuj problém.

Definuj specifikace problému konfigurací parametrů ve slovníku `qubo_settings`.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

### 3. Definuj nastavení optimalizátoru a ansatzu. (Volitelné)
Volitelně definuj konkrétní požadavky na optimalizační proces, včetně výběru optimalizátoru a jeho parametrů, a také specifikaci primitivu a jeho konfigurací.

Pro Tailored Ansatz byl zvolený počet jedinců populace (`population_size`) na základě předchozích experimentů, které ukázaly, že tato hodnota přináší stabilní a efektivní optimalizaci.

V případě Real Amplitudes Ansatzu lze použít lineární vztah mezi ``population_size`` a počtem qubitů v Circuit. Jako přibližné pravidlo se doporučuje používat **minimálně** ``population_size ~ 0.8 * n_qubits`` pro ansatz `real_amplitudes`.

Očekává se, že Optimized Real Amplitudes bude mít lepší optimalizační výkon než ansatz Real Amplitudes. Počet proměnných k optimalizaci v tomto ansatzu však roste mnohem rychleji než v případě Real Amplitudes (viz [článek](https://arxiv.org/pdf/2412.19150)). Proto u rozsáhlých problémů vyžaduje Optimized Real Amplitudes více spuštění Circuit. Optimized Real Amplitudes bude pravděpodobně užitečný pro problémy vyžadující až 100 qubitů, ale při nastavování parametru ``population_size`` je doporučeno postupovat obezřetně. Jako příklad tohoto škálování ``population_size`` předchozí tabulka ukazuje, že u problému s 84 qubity vyžaduje Optimize Real Amplitudes ``population_size`` 120, zatímco pro problém s 56 qubity postačuje ``population_size`` 40.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

Je také možné zvolit konkrétní ansatz. Následující kód používá ansatz ``'Tailored'``.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 4. Spusť problém.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

### 5. Načtení výsledků.
Jak je uvedeno v části [Výstup](#output), funkce vrací slovník s investičními trajektoriemi seřazenými od nejnižší po nejvyšší podle hodnoty jejich účelové funkce. Tato sada výsledků umožňuje identifikovat trajektorii s nejnižšími náklady a příslušná investiční hodnocení. Navíc umožňuje analýzu různých trajektorií, čímž usnadňuje výběr těch, které nejlépe odpovídají konkrétním potřebám nebo cílům. Tato flexibilita zajišťuje, že volby lze přizpůsobit různým preferencím nebo scénářům.
Začni tím, že zobrazíš výslednou strategii, která dosáhla nejnižších nákladů účelové funkce nalezených v průběhu procesu.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


### 6. Performance analysis

Last, analyze the performance of your optimization application. Specifically, compare your results, obtained in the previous example, against a random baseline to assess the effectiveness of our approach. If the quantum algorithm demonstrably and consistently produces results with lower cost values, it indicates an effective optimization process.

The figure presents the probability distributions of the objective costs. To generate these distributions, take the list of objective costs from the function result and count the occurrences of each cost value (values rounded to the second decimal place). Then, update the count column accordingly by joining counts of identical rounded values. Note that, for better visual comparison, the occurrence counts have been normalized so that each distribution is displayed between 0 and 1.

![Visualization of the solution of the optimization](../docs/images/guides/global-data-quantum-optimizer/cost_distribution.svg)

As shown in the figure (blue solid line), the cost distribution for our Variational Quantum Eigensolver (post-processed with SQD) approach is sharply concentrated at lower objective cost values, indicating good optimization performance. In contrast, the noisy baseline exhibits a broader distribution, centered around higher cost values. The gray dashed vertical line represents the mean value of the random distribution, further highlighting the consistency of the function in returning optimized investment strategies. For an additional comparison, the black dashed line in the figure corresponds to the solution obtained with the Gurobi optimizer (free version). All these results are further explored in the benchmarks below for the "Mixed Assets" example evaluated with the "Tailored" ansatz.

# Benchmarks

This function was tested under different configurations of resolution qubits, ansatz circuits, and groupings of assets from various sectors: a mix of different assets (Set 1), oil derivatives (Set 2), and IBEX35 (Set 3). See more details in the following table.

| Set       | Date       | Assets                                                                 |
|-----------|------------|------------------------------------------------------------------------|
| **Set 1** | 01/01/2023 | 8801.T, CL=F, GBPJPY=X, ITX.MC, META, TMBMKDE-10Y, XS2239553048         |
| **Set 2** | 01/06/2023 | CL=F, BZ=F, HO=F, NG=F, XOM, RB=F, 2222.SR                               |
| **Set 3** | 01/11/2022 | ACS.MC, ITX.MC, FER.MC, ELE.MC, SCYR.MC, AENA.MC, AMS.MC                |

Two key metrics were used to evaluate solution quality.
1. The objective cost, which measures optimization efficiency by comparing the cost function value from each experiment with results from Gurobi (free version).
2. The Sharpe ratio, which captures the risk-adjusted return of each portfolio, offering insight into the financial performance of the solutions.

Together, these metrics benchmark both computational and financial aspects of the quantum-generated portfolios.


| Example             | Qubits | Ansatz                  | Depth | Runtime Usage (s) | Total usage (s) | Objective cost | Sharpe | Gurobi objective cost | Gurobi Sharpe |
|-------------------------------|--------|-------------------------------|--------|-------------------|------------------|----------------|--------|------------------------|----------------|
| Mixed Assets (Set 1, 4 time steps, 4-bit)   | 112    | Tailored                      | 83     | 12735             | 13095            | -3.78          | 24.82  | -4.25                  | 24.71          |
| Mixed Assets (Set 1,4 time steps, 4 time steps, 4-bit)   | 112    | Real Amplitudes               | 359    | 11739             | 11903            | -3.39          | 23.64  | -4.25                  | 24.71          |
| Oil Derivatives (Set 2, 4 time steps, 3-bit)| 84     | Optimized Real Amplitudes     | 78     | 6180              | 6350             | -3.73          | 19.13  | -4.19                  | 21.71          |
| IBEX35 (Set 3, 4 time steps, 2-bit)         | 56     | Optimized Real Amplitudes     | 96     | 3314              | 3523             | -3.67          | 14.48  | -4.11                  | 16.44          |



Results show that the quantum optimizer, with problem-specific ansatzes, effectively identifies efficient investment strategies across various portfolio types.

Below we detail both the population size and the number of generations specified in the `optimizer_options` dictionary. All other parameters were set to their default values.

| **Example**                | ``population_size`` | ``num_generations``   |
|----------------------------|----------------------|----------------------|
| Mixed Assets Portfolio     | 90                   | 20                   |
| Mixed Assets Portfolio     | 92                   | 20                   |
| Oil Derivatives Portfolio  | 120                  | 20                   |
| IBEX35 Portfolio           | 40                   | 20                   |

The number of generations was set to 20, as this value was found to be sufficient to reach convergence. Additionally, the default values for the optimizer's internal parameters were left unchanged, as they consistently provided good performance and are generally recommended by the literature and implementation guidelines.

# Get support
If you need help, you can send an email to qpo.support@globaldataquantum.com. In your message, provide the function job ID.

## Next steps

<Admonition type="note" title="Recommendations">
*   Read [the associated research paper](https://arxiv.org/pdf/2412.19150).
*   Request access to the function by filling in [this form](https://www.globaldataquantum.com/en/quantum-portfolio-optimizer/#form).
*   Try the [Dynamic Portfolio Optimization](/docs/tutorials/global-data-quantum-optimizer) tutorial.

</Admonition>